<a href="https://colab.research.google.com/github/AmrKhaled545/Retail-Demand-Forecasting-Project/blob/main/Retail_Demand_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retail Demand Forecasting — Walmart M5 Sales Pipeline

## Project Overview

This notebook implements an end-to-end retail demand forecasting pipeline
built on the M5 Forecasting (Walmart) dataset. It covers data ingestion,
memory-efficient feature engineering at scale, exploratory analysis of sales
and pricing patterns, model training and comparison, and a 28-day recursive
forecast that powers the companion Streamlit application.

## Design Principles

- **Scales independently of dataset size.** Product selection (`TOP_N`) and
  processing batch size (`CHUNK_SIZE`) are decoupled — peak memory usage is
  governed only by `CHUNK_SIZE`, so `TOP_N` can be raised from 15 to 1,500+
  without changing how the pipeline runs.
- **Category-balanced by design.** Products are selected proportionally
  across categories (`FOODS`, `HOUSEHOLD`, `HOBBIES`) rather than by raw
  sales volume alone, so the model trains on a representative product mix
  instead of being dominated by the largest category.
- **Memory-aware at every stage.** Numeric and categorical columns are
  downcast to the smallest safe dtype throughout, EDA statistics are
  accumulated incrementally rather than computed on a fully materialized
  table, and intermediate frames are released as soon as they're no longer
  needed.
- **Model choice is evidence-based.** XGBoost and LightGBM are trained and
  compared on identical features and metrics; an optional LSTM baseline is
  included for completeness, with a documented rationale for why gradient
  boosting is the stronger default for this dataset.

## Notebook Structure

1. Data loading & category-balanced product selection
2. Chunked feature engineering pipeline
3. Exploratory Data Analysis
4. Data cleaning & preprocessing (recap)
5. Time-series feature engineering (recap)
6. Chronological train/test split
7. Baseline model: XGBoost
8. Model comparison: LightGBM & optional deep learning baseline
9. Recursive 28-day forecasting & deployment artifacts
10. Deployment via Streamlit


In [1]:
# Data Analysis Libraries
import pandas as pd
import numpy as np
import gc
import warnings

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Modeling Libraries
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore', category=FutureWarning)


In [2]:
def reduce_mem_usage(df, verbose=False):
    """Downcast numeric columns to the smallest dtype that fits, in place."""
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage: {start_mem:.2f} MB -> {end_mem:.2f} MB '
              f'({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df


def print_memory_usage(label=''):
    """Best-effort process RAM report. Falls back quietly if psutil isn't installed."""
    try:
        import psutil, os
        rss_mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        print(f'[memory] {label}: {rss_mb:.1f} MB in use')
    except ImportError:
        pass


## Step 1 — Data Loading & Product Selection

`calendar.csv` and `sell_prices.csv` are loaded directly. `sales_train_validation.csv`
is loaded in its original *wide* format — one row per product-store
combination with one column per day — since reshaping it before selecting
products would force the full dataset into memory unnecessarily. Categorical
columns are cast to compact dtypes immediately, before any melting or
merging takes place.


In [3]:
# Getting the uploaded data from Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DIR = '/content/drive/MyDrive/DEPI/DEPI_Grad_Project/data/'  # <-- change to your path, e.g. '/content/drive/MyDrive/DEPI/DEPI_Grad_Project/data/'

calendar = pd.read_csv(DATA_DIR + 'calendar.csv', parse_dates=['date'])
sell_prices = pd.read_csv(DATA_DIR + 'sell_prices.csv')

for c in ['weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    if c in calendar.columns:
        calendar[c] = calendar[c].astype('category')

id_dtypes = {'id': 'category', 'item_id': 'category', 'dept_id': 'category',
             'cat_id': 'category', 'store_id': 'category', 'state_id': 'category'}
sales_wide = pd.read_csv(DATA_DIR + 'sales_train_validation.csv', dtype=id_dtypes)

d_cols = [c for c in sales_wide.columns if c.startswith('d_')]
sales_wide[d_cols] = sales_wide[d_cols].astype('int16')

print('Raw wide shape:', sales_wide.shape)
print_memory_usage('after loading raw files')


Raw wide shape: (30490, 1919)
[memory] after loading raw files: 996.0 MB in use
